In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
client = OpenAI()

print("Setup complete")

Setup complete


In [2]:
import gradio as gr

def summarize_pros_cons(topic, num_points):
    prompt = f"List {num_points} pros and cons of: {topic}"
    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

demo = gr.Interface(
    fn=summarize_pros_cons,
    inputs=["text", gr.Slider(1, 5, value=3, step=1)],
    outputs="text"
)
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [3]:
def chat_stream(message):
    stream = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": message}],
        stream=True
    )

    partial_reply = ""
    for chunk in stream:
        content = chunk.choices[0].delta.content
        if content:
            partial_reply += content
            yield partial_reply

demo = gr.Interface(fn=chat_stream, inputs="text", outputs="text")
demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [4]:
import gradio as gr
from anthropic import Anthropic

anthropic_client = Anthropic()

def stream_gpt(message):
    stream = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": message}],
        stream=True
    )
    reply = ""
    for chunk in stream:
        content = chunk.choices[0].delta.content
        if content:
            reply += content
            yield reply

def stream_claude(message):
    reply = ""
    with anthropic_client.messages.stream(
        model="claude-sonnet-4-5",
        max_tokens=500,
        messages=[{"role": "user", "content": message}]
    ) as stream:
        for text in stream.text_stream:
            reply += text
            yield reply

with gr.Blocks() as demo:
    prompt = gr.Textbox(label="Your question")
    with gr.Row():
        gpt_output = gr.Textbox(label="GPT-5")
        claude_output = gr.Textbox(label="Claude")
    submit = gr.Button("Ask both")
    submit.click(stream_gpt, inputs=prompt, outputs=gpt_output)
    submit.click(stream_claude, inputs=prompt, outputs=claude_output)

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
